# Week 08 - Train the keyword-spotting model on Google Colab

Runs the same scripts as your laptop (`train.py`, `evaluate.py`, `verify_c_features.py`, `export_c_array.py`).
A CPU runtime is enough; a GPU runtime is faster for large datasets.

1. Zip your dataset folder so the zip contains `dataset/<label>/*.wav` and upload it in step 2,
   **or** set `USE_SYNTHETIC = True` to test the pipeline without recordings.
2. Run all cells. At the end, download `week08_export.zip` and copy
   `model_data.cc` + `kws_labels.h` into `labs/firmware/kws-inference/main/`.

In [ ]:
!git clone --depth 1 https://github.com/drsanti/kmutt-nd69.git
%cd kmutt-nd69/nd1/docs/week08/labs/python
!pip install -q ai-edge-litert

In [ ]:
USE_SYNTHETIC = False   # True = generate a synthetic plumbing-test dataset instead of uploading

if USE_SYNTHETIC:
    !python make_synthetic_dataset.py --out dataset --per-class 60
else:
    from google.colab import files
    up = files.upload()                      # choose dataset.zip
    zip_name = next(iter(up))
    !unzip -q -o "$zip_name"
!ls dataset

In [ ]:
!python train.py --data dataset --out artifacts --epochs 40

In [ ]:
!python evaluate.py --artifacts artifacts --data dataset
# Domain-shift check: board clips only
!python evaluate.py --artifacts artifacts --data dataset --device board || echo "no board clips in the test split"

In [ ]:
# C features (firmware) must match the Python features used for training
!python verify_c_features.py --data dataset --max-clips 30

In [ ]:
!python export_c_array.py --model artifacts/model_int8.tflite --out-dir export
!cp artifacts/report.md artifacts/meta.json artifacts/*.tflite export/
!cd export && zip -q -r ../week08_export.zip .
from google.colab import files
files.download("week08_export.zip")